In [1]:
import torchxrayvision as xrv
import torch, torchvision
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt

# Projection algorithms
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
import umap.umap_ as umap
import pacmap

In [2]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.widgets import RectangleSelector

In [3]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import numpy as np
import pandas as pd

### Constants and Configs

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed seed for reproducibility
RANDOM_STATE = 42

In [5]:
PATHOLOGIES = [
    "Lung Opacity",
    "Fracture",
    # TODO: check if we should add this or not
    "Nodule/Mass",
    "Pneumothorax"
]

### Load and Prepare

In [6]:
def load_dataset(image_path: str = "./nih_data_small", datasetClass = xrv.datasets.NIH_Google_Dataset):
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    d_nih = datasetClass(imgpath=image_path, transform=transform, views=["PA", "AP"])
    print(f'found {len(d_nih)} images')
    return d_nih

In [ ]:
NIH_GOOGLE_DATASET = load_dataset()
DATASET_PATHOLOGIES = list(NIH_GOOGLE_DATASET.pathologies)
DATASET_LABEL_INDICES = {
        name: DATASET_PATHOLOGIES.index(name)
        for name in PATHOLOGIES
}
DATASET_INDICES_LABEL = {DATASET_LABEL_INDICES[l]:l for l in DATASET_LABEL_INDICES}

found 1695 images


In [8]:
def get_patient_gender(dataset):
    metadata = getattr(dataset, "csv", None)

    def normalize_gender(value):
        value = str(value).strip().lower()

        if value == 'm':
            return 0.0
        if value == 'f':
            return 1.0
        return 2.0

    return metadata["Patient Gender"].map(normalize_gender).to_numpy()

In [9]:
NIH_GOOGLE_GENDER_LIST = get_patient_gender(NIH_GOOGLE_DATASET)

In [10]:
def get_patient_age(dataset):
    metadata = getattr(dataset, "csv", None)

    def parse_age(value):
        value = str(value).strip().lower()

        return int(value)

    return metadata["Patient Age"].map(parse_age).to_numpy()

In [11]:
NIH_GOOGLE_AGE_LIST = get_patient_age(NIH_GOOGLE_DATASET)

In [12]:
NIH_GOOGLE_DATASET.csv

,Patient ID,Image Index,Finding Labels,Follow-up #,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Fracture,Pneumothorax,Airspace opacity,Nodule or mass,Set Id,view
0,13,00000013_008.png,No Finding,8,60,M,AP,3056,2544,0.139000,0.139000,NO,NO,YES,NO,test,AP
1,32,00000032_002.png,No Finding,2,55,F,AP,2500,2048,0.168000,0.168000,NO,NO,YES,NO,test,AP
2,116,00000116_010.png,Cardiomegaly|Effusion,10,75,F,AP,2500,2048,0.168000,0.168000,NO,NO,YES,NO,test,AP
3,119,00000119_001.png,Atelectasis,1,56,M,AP,2500,2048,0.168000,0.168000,NO,NO,NO,NO,val,AP
4,134,00000134_000.png,Infiltration,0,76,F,AP,2048,2500,0.168000,0.168000,NO,NO,YES,NO,val,AP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1690,30709,00030709_002.png,No Finding,2,48,M,AP,3056,2544,0.139000,0.139000,NO,NO,NO,NO,test,AP
1691,30719,00030719_000.png,No Finding,0,54,M,PA,2021,2021,0.194311,0.194311,NO,NO,NO,YES,test,PA
1692,30752,00030752_000.png,No Finding,0,64,F,AP,3056,2544,0.139000,0.139000,NO,NO,NO,NO,test,AP
1693,30759,00030759_000.png,Consolidation,0,51,M,PA,2021,2021,0.194311,0.194311,NO,NO,NO,NO,val,PA


In [13]:
DATASET_INDICES_LABEL

{0: 'Lung Opacity', 1: 'Fracture', 2: 'Nodule/Mass', 3: 'Pneumothorax'}

In [14]:
def create_image_feature_space_and_label(dataset, dataset_indices_label = DATASET_INDICES_LABEL):
    image_features_list = []
    labels_list = []
    for i in range(len(dataset)):
        sample = dataset[i]
        label = np.asarray(sample["lab"]).reshape(-1)

        img = np.asarray(sample["img"]).squeeze()
        image_features_list.append(
                        img.astype(np.float32).reshape(-1)
            )
            
        labels_list.append(
                    label
                )

    image_features = np.stack(
        image_features_list,
        axis=0
    )
    labels = np.array(labels_list)
    return image_features, labels


In [15]:
NIH_GOOGLE_IMAGE_FEATURES, NIH_GOOGLE_LABELS_LIST = create_image_feature_space_and_label(NIH_GOOGLE_DATASET)

### Load model

In [16]:
def load_model(model_weights: str = "densenet121-res224-chex"):
    model = xrv.models.DenseNet(weights=model_weights)
    model.eval()
    return model.to(DEVICE)

In [17]:
# MODEL_NAME = "densenet121-res224-all"
MODEL_NAME = "densenet121-res224-chex"
DENSENET121_RES244_CHEX_MODEL = load_model(model_weights=MODEL_NAME)
DENSENET121_RES244_CHEX_MODEL.eval()

XRV-DenseNet121-densenet121-res224-chex

In [18]:
def get_model_to_dataset_indices(model = DENSENET121_RES244_CHEX_MODEL, dataset_label_indices=DATASET_LABEL_INDICES):
    return {
        model_idx: dataset_label_indices[pathology]
        for model_idx, pathology in enumerate(model.pathologies)
        if pathology in dataset_label_indices
    }   

In [47]:
MODEL_TO_DATASET_INDICES = get_model_to_dataset_indices()
DATASET_TO_MODEL_INDICES = {MODEL_TO_DATASET_INDICES[key]:key for key in MODEL_TO_DATASET_INDICES}

### Prediction of the model

In [20]:
def calculate_model_predictions_score(
    feature_space,
    model,
    batch_size=32
):
    model.eval()

    # Convert flattened images to (N, 1, 224, 224)
    images = torch.as_tensor(feature_space, dtype=torch.float32)
    images = images.reshape(-1, 1, 224, 224)

    # Run inference in batches
    all_outputs = []

    with torch.no_grad():
        for start_idx in range(0, len(images), batch_size):
            end_idx = start_idx + batch_size

            batch = images[start_idx:end_idx].to(DEVICE)

            batch_outputs = model(batch)

            # Move each batch back to CPU immediately
            batch_outputs = batch_outputs.cpu().numpy()


            all_outputs.append(batch_outputs)

    # Combine all batches
    selected_outputs = np.concatenate(all_outputs, axis=0)
    return selected_outputs

In [21]:
MODEL_SCORE_PREDICTIONS = calculate_model_predictions_score(NIH_GOOGLE_IMAGE_FEATURES,
                                                       DENSENET121_RES244_CHEX_MODEL)

### Downprojection

In [22]:
class Downprojector:

    def __init__(self, image_features):
        self.projection_cache = {}
        self.image_features = image_features


    def get_feature_matrix(self, feature_space):
        return self.image_features

    """
    PCA is used as a preprocessing step for the non-linear
        projection techniques.
    """
    def prepare_pca_features(self, feature_space):
        cache_key = f"{feature_space}_pca"
         
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]
         
        X = self.get_feature_matrix(feature_space)
         
             # Keep at most 50 dimensions before expensive
             # non-linear dimensionality reduction.
        n_components = min(
                 50,
                 X.shape[0],
                 X.shape[1]
             )
         
        pca = PCA(
                 n_components=n_components,
                 random_state=RANDOM_STATE
        )
         
        X_pca = pca.fit_transform(X)
         
        self.projection_cache[cache_key] = {
                 "data": X_pca,
                 "pca": pca
             }
         
        return self.projection_cache[cache_key]

    def calculate_projection(self, feature_space: str, method: str,
                             target_labels=None,
                             label_name=None,
                             dataset_label_to_index_map = DATASET_LABEL_INDICES,):
        
        cache_key = (
                f"{feature_space}_{method}"
            ) if label_name is None else f"{feature_space}_{method}_{label_name}"
        
        if cache_key in self.projection_cache:
            return self.projection_cache[cache_key]


        if method == "PCA":

            X = self.get_feature_matrix(feature_space)

            reducer = PCA(
                n_components=2,
                random_state=RANDOM_STATE
            )

            embedding = reducer.fit_transform(X)
        elif method == "t-SNE":
        
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]
        
                # t-SNE requires perplexity < number of samples.
            perplexity = min(
                    30,
                    max(5, (X_pca.shape[0] - 1) // 3)
                )
        
            reducer = TSNE(
                    n_components=2,
                    perplexity=perplexity,
                    init="pca",
                    learning_rate="auto",
                    random_state=RANDOM_STATE
                )
        
            embedding = reducer.fit_transform(
                    X_pca
                )
        elif method == "UMAP":
            X_pca = self.prepare_pca_features(
                    feature_space
                )["data"]

            n_neighbors = min(
                    15,
                    max(2, X_pca.shape[0] - 1)
                )

            reducer = umap.UMAP(
                    n_components=2,
                    n_neighbors=n_neighbors,
                    min_dist=0.1,
                    metric="euclidean",
                    random_state=RANDOM_STATE
                )

            embedding = reducer.fit_transform(
                    X_pca
                )
        # Supervised UMAP
        elif method == "S-UMAP":

            if target_labels is None:
                raise ValueError('Supervised UMAP needs target labels')

            if label_name is None:
                raise ValueError('Supervised UMAP needs a label name / pathologie')
            
            X_pca = self.prepare_pca_features(
                                feature_space
                            )["data"]
            
            n_neighbors = min(
                                15,
                                max(2, X_pca.shape[0] - 1)
                            )
            
            reducer = umap.UMAP(
                                n_components=2,
                                n_neighbors=n_neighbors,
                                min_dist=0.1,
                                metric="euclidean",
                                random_state=RANDOM_STATE
                            )

            target_index = dataset_label_to_index_map[label_name]

            Y = target_labels[:, target_index]
            
            embedding = reducer.fit_transform(
                                X_pca,
                                Y
                            )
        else:
            raise ValueError(
                f"Unknown projection method: {method}"
                )
        self.projection_cache[cache_key] = embedding
        return embedding

### Run first visualization

In [23]:
NIH_GOOGLE_GENDER_LIST

array([0., 1., 1., ..., 1., 0., 1.], shape=(1695,))

In [24]:
def visualize_space(feature_space, projection_method,
                    label_name="Pneumothorax",
                    labels_list=NIH_GOOGLE_LABELS_LIST,
                    dataset_label_indices = DATASET_LABEL_INDICES,
                    dataset_gender_list = NIH_GOOGLE_GENDER_LIST):
    label_index = dataset_label_indices[label_name]
    label_pathologie_label_output = labels_list[:, label_index]

    idx_pathologie = label_pathologie_label_output == 1
    idx_non_pathologie = label_pathologie_label_output == 0


    idx_pathologie_male = dataset_gender_list[idx_pathologie] == 0
    idx_pathologie_female = dataset_gender_list[idx_pathologie] == 1

    idx_non_pathologie_male = dataset_gender_list[idx_non_pathologie] == 0
    idx_non_pathologie_female = dataset_gender_list[idx_non_pathologie] == 1

    fig, ax = plt.subplots(figsize=(11, 8))

    ax.scatter(feature_space[idx_pathologie,0][idx_pathologie_male], 
               feature_space[idx_pathologie,1][idx_pathologie_male], label=label_name, c='red',
               marker='o')
    ax.scatter(feature_space[idx_pathologie,0][idx_pathologie_female], 
               feature_space[idx_pathologie,1][idx_pathologie_female], label=label_name, c='red',
               marker='^')

    ax.scatter(feature_space[idx_non_pathologie,0][idx_non_pathologie_male], feature_space[idx_non_pathologie,1][idx_non_pathologie_male], label='Not effected', c='blue',
                alpha=0.1, marker='o')
    ax.scatter(feature_space[idx_non_pathologie,0][idx_non_pathologie_female], feature_space[idx_non_pathologie,1][idx_non_pathologie_female], label='Not effected', c='blue',
                alpha=0.1, marker='^')

    ax.set_title(f"{projection_method} Projection Colored")
    ax.set_xlabel(f"{projection_method} 1")
    ax.set_ylabel(f"{projection_method} 2")
    ax.legend()
    plt.show()

In [25]:
downprojector = Downprojector(NIH_GOOGLE_IMAGE_FEATURES)

In [49]:
def create_viewer(downprojector: Downprojector, visualization_fn):

    plot_output = widgets.Output()

    plot_button = widgets.Button(
        description="Create selection plot",
        button_style="primary",
        icon="refresh"
    )

    pathology_dropdown = widgets.Dropdown(
        options=PATHOLOGIES,
        value=PATHOLOGIES[0],
        description="Pathology:",
        layout=widgets.Layout(width="350px")
    )

    projection_dropdown = widgets.Dropdown(
        options=[
            "PCA",
            "t-SNE",
            "UMAP",
            "S-UMAP"
        ],
        value="PCA",
        description="Projection:",
        layout=widgets.Layout(width="300px")
    )

    def create_selection_plot(_=None):

        with plot_output:
            clear_output(wait=True)

            pathology = pathology_dropdown.value
            method = projection_dropdown.value

            target_labels = None
            label_name = None
            
            if method == 'S-UMAP':
                target_labels = NIH_GOOGLE_LABELS_LIST
                label_name = pathology

            # Calculate the requested projection if it has not already
            # been calculated.
            embedding = downprojector.calculate_projection(
                "feature",
                method,
                target_labels=target_labels,
                label_name = label_name
            )
            visualization_fn(embedding, method, pathology)
            

    plot_button.on_click(create_selection_plot)

    display(
    widgets.VBox([
        widgets.HBox([
            projection_dropdown
        ]),
        widgets.HBox([
            pathology_dropdown
        ]),
        plot_button,
        plot_output,
    ])
)

In [27]:
create_viewer(downprojector, visualize_space)

### Visualize predictions


In [28]:
def visualize_space_prediction_scores(feature_space, projection_method,label_name="Pneumothorax", 
                                      predictions=MODEL_SCORE_PREDICTIONS,
                    dataset_label_indices = DATASET_LABEL_INDICES,
                    dataset_to_model_indices=DATASET_TO_MODEL_INDICES):

    # Find the prediction column for this finding
    dataset_label_index = dataset_label_indices[label_name]
   
    model_label_index = dataset_to_model_indices[dataset_label_index]

    # Get the model score for each image
    scores = predictions[:, model_label_index]

    plt.figure(figsize=(10, 7))

    scatter = plt.scatter(
        feature_space[:, 0],
        feature_space[:, 1],
        c=scores,
        cmap="viridis",
        alpha=0.8,
        s=20
    )

    plt.colorbar(scatter, label=f"Model score: {label_name}")
    plt.title(f"{projection_method} Projection Colored by {label_name} Score")
    plt.xlabel(f"{projection_method} 1")
    plt.ylabel(f"{projection_method} 2")
    plt.show()

In [ ]:
create_viewer(downprojector, visualize_space_prediction_scores)

### Compare Prediction with Dataset

In [30]:
def convert_to_dataset_label_list(model_prediction_score, dataset_labels_list = NIH_GOOGLE_LABELS_LIST,
                                  dataset_to_model_indices=DATASET_TO_MODEL_INDICES
                                  ):
    new_prection_list = np.zeros_like(dataset_labels_list)
    for selectedDatasetIndex in dataset_to_model_indices:
        selectedModelIndex = dataset_to_model_indices[selectedDatasetIndex]
        new_prection_list[:, selectedDatasetIndex] = model_prediction_score[:, selectedModelIndex]
    return new_prection_list

MODEL_SCORE_PREDICTIONS_CONVERTED = convert_to_dataset_label_list(MODEL_SCORE_PREDICTIONS)


In [31]:
def calulcate_metrics_for_model(
    model_prediction,
    dataset_labels_list=NIH_GOOGLE_LABELS_LIST,
    threshold=0.5,
    dataset_label_to_indicies = DATASET_LABEL_INDICES
):

    # Convert scores into binary predictions
    binary_prediction = (model_prediction >= threshold).astype(int)


    # Store metrics for every label
    metrics = []

    for label in PATHOLOGIES:
        label_idx = dataset_label_to_indicies[label]
        # Ground truth
        y_true = dataset_labels_list[:, label_idx]

        # Prediction
        y_pred = binary_prediction[:, label_idx]

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1]
        ).ravel()

        # Metrics
        precision = precision_score(
            y_true, y_pred, zero_division=0
        )

        recall = recall_score(
            y_true, y_pred, zero_division=0
        )

        f1 = f1_score(
            y_true, y_pred, zero_division=0
        )

        accuracy = accuracy_score(
            y_true, y_pred
        )

        # Specificity = TN / (TN + FP)
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0
            else 0
        )

        # False-positive rate
        fpr = (
            fp / (fp + tn)
            if (fp + tn) > 0
            else 0
        )

        # False-negative rate
        fnr = (
            fn / (fn + tp)
            if (fn + tp) > 0
            else 0
        )

        # ROC-AUC can fail if only one class exists
        try:
            auc = roc_auc_score(
                y_true,
                model_prediction[:, label_idx]
            )
        except ValueError:
            auc = np.nan


        metrics.append({
            "label": label,
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "specificity": specificity,
            "FPR": fpr,
            "FNR": fnr,
            "F1": f1,
            "ROC_AUC": auc
        })

    metrics_df = pd.DataFrame(metrics)

    # Overall metrics
    overall_metrics = {
        "macro_accuracy": metrics_df["accuracy"].mean(),
        "macro_precision": metrics_df["precision"].mean(),
        "macro_recall": metrics_df["recall"].mean(),
        "macro_specificity": metrics_df["specificity"].mean(),
        "macro_F1": metrics_df["F1"].mean(),
        "macro_ROC_AUC": metrics_df["ROC_AUC"].mean(),

        "micro_precision": precision_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        ),

        "micro_recall": recall_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        ),

        "micro_F1": f1_score(
            dataset_labels_list.flatten(),
            binary_prediction.flatten(),
            zero_division=0
        )
    }

    return metrics_df, overall_metrics

In [32]:
calulcate_metrics_for_model(MODEL_SCORE_PREDICTIONS_CONVERTED, NIH_GOOGLE_LABELS_LIST, threshold=0.7)

(          label   TP    TN   FP   FN  accuracy  precision    recall  \
 0  Lung Opacity  463   829  265  138  0.762242   0.635989  0.770383   
 1      Fracture    9  1450  185   51  0.860767   0.046392  0.150000   
 2   Nodule/Mass    0  1463    0  232  0.863127   0.000000  0.000000   
 3  Pneumothorax    0  1623    0   72  0.957522   0.000000  0.000000   
 
    specificity      FPR       FNR        F1   ROC_AUC  
 0      0.75777  0.24223  0.229617  0.696764  0.845164  
 1      0.88685  0.11315  0.850000  0.070866  0.541600  
 2      1.00000  0.00000  1.000000  0.000000  0.500000  
 3      1.00000  0.00000  1.000000  0.000000  0.696250  ,
 {'macro_accuracy': np.float64(0.8609144542772862),
  'macro_precision': np.float64(0.17059519089158265),
  'macro_recall': np.float64(0.23009567387687188),
  'macro_specificity': np.float64(0.9111549513890054),
  'macro_F1': np.float64(0.19190765657678793),
  'macro_ROC_AUC': np.float64(0.6457535248445816),
  'micro_precision': 0.5119305856832972,
 

### Visualize Prediction

In [33]:
def calculate_age_error_rates(
    pathology,
    ages=NIH_GOOGLE_AGE_LIST,
    ground_truth=NIH_GOOGLE_LABELS_LIST,
    predictions=MODEL_SCORE_PREDICTIONS_CONVERTED,
    threshold=0.7,
    min_samples=10
):

    label_idx = DATASET_LABEL_INDICES[pathology]

    # Extract the data for the selected pathology
    y_true = np.asarray(ground_truth[:, label_idx])
    y_scores = np.asarray(predictions[:, label_idx])
    ages = np.asarray(ages, dtype=float)

    # Convert model scores into binary predictions
    y_pred = (y_scores >= threshold).astype(int)


    df = pd.DataFrame({
        "age": ages,
        "true": y_true.astype(int),
        "pred": y_pred
    })

    # Ten-year age groups
    age_bins = list(range(0, 111, 10))
    age_labels = [
        f"{start}-{start + 9}"
        for start in age_bins[:-1]
    ]

    df["age_group"] = pd.cut(
        df["age"],
        bins=age_bins,
        labels=age_labels,
        right=False
    )

    # Exclude ages outside the defined bins
    df = df.dropna(subset=["age_group"])

    results = []

    for age_group, group in df.groupby(
        "age_group", observed=False
    ):
        y_t = group["true"].to_numpy()
        y_p = group["pred"].to_numpy()

        # Confusion-matrix counts
        tp = np.sum((y_t == 1) & (y_p == 1))
        tn = np.sum((y_t == 0) & (y_p == 0))
        fp = np.sum((y_t == 0) & (y_p == 1))
        fn = np.sum((y_t == 1) & (y_p == 0))

        n_positive = tp + fn
        n_negative = tn + fp

        # Rates have different denominators
        fpr = fp / n_negative if n_negative > 0 else np.nan
        fnr = fn / n_positive if n_positive > 0 else np.nan

        # Avoid interpreting rates based on very few cases
        if n_negative < min_samples:
            fpr = np.nan

        if n_positive < min_samples:
            fnr = np.nan

        results.append({
            "age_group": str(age_group),
            "age_midpoint": int(age_group.split("-")[0]) + 5,
            "n_images": len(group),
            "n_positive": n_positive,
            "n_negative": n_negative,
            "FP": fp,
            "FN": fn,
            "FPR": fpr,
            "FNR": fnr
        })

    return pd.DataFrame(results)


In [34]:
def plot_age_error_rates(pathology, threshold=0.7):
    results = calculate_age_error_rates(
        pathology=pathology,
        threshold=threshold
    )
    plt.figure()
    fig, ax = plt.subplots(figsize=(10, 6))

    # False-positive rate
    ax.plot(
        results["age_midpoint"],
        results["FPR"] * 100,
        marker="o",
        linewidth=2,
        label="False-positive rate (FP / (FP + TN))"
    )

    # False-negative rate
    ax.plot(
        results["age_midpoint"],
        results["FNR"] * 100,
        marker="s",
        linewidth=2,
        label="False-negative rate (FN / (FN + TP))"
    )

    ax.set_title(
        f"Model Error Rates by Age — {pathology}"
    )
    ax.set_xlabel("Patient age")
    ax.set_ylabel("Error rate (%)")

    ax.set_xticks(results["age_midpoint"])
    ax.set_xticklabels(results["age_group"], rotation=45)

    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()

    # Display the underlying numbers for interpretation
    display_columns = [
        "age_group",
        "n_images",
        "n_positive",
        "n_negative",
        "FP",
        "FN",
        "FPR",
        "FNR"
    ]

    display(
        results[display_columns].round(3)
    )

    return results

In [35]:
class AgeErrorRateViewer:

    def __init__(self, visualization_fn, threshold=0.5):
        self.visualization_fn = visualization_fn
        self.threshold = threshold

    def show(self):
        
        plot_output = widgets.Output()

        plot_button = widgets.Button(
            description="Create plot",
            button_style="primary",
            icon="refresh"
        )

        pathology_dropdown = widgets.Dropdown(
            options=PATHOLOGIES,
            value=PATHOLOGIES[0],
            description="Pathology:",
            layout=widgets.Layout(width="350px")
        )

        threshold_input = widgets.FloatText(
            value=self.threshold
        )

        def create_plot(_=None):

            with plot_output:
                clear_output(wait=True)

                pathology = pathology_dropdown.value  
                
                self.threshold = threshold_input.value      

                self.visualization_fn(pathology, threshold= self.threshold)
                

        plot_button.on_click(create_plot)

        display(
        widgets.VBox([
            widgets.HBox([
                pathology_dropdown
            ]),
            widgets.HBox([
                            threshold_input
                        ]),
            plot_button,
            plot_output,
        ])
    )
    

In [36]:
ageErrorViewer = AgeErrorRateViewer(plot_age_error_rates)
ageErrorViewer.show()

In [37]:
def visualize_comparison_groundtruth_with_prediction(feature_space, projection_method,label_name="Pneumothorax", 
                                      predictions=MODEL_SCORE_PREDICTIONS,
                                      dataset_labels = NIH_GOOGLE_LABELS_LIST,
                    dataset_label_indices = DATASET_LABEL_INDICES,
                    dataset_to_model_indices=DATASET_TO_MODEL_INDICES):

    # Find the prediction column for this finding
    dataset_label_index = dataset_label_indices[label_name]
   
    model_label_index = dataset_to_model_indices[dataset_label_index]

    # Get the model score for each image
    scores = predictions[:, model_label_index]

    plt.figure(figsize=(10, 7))

    sizes = 20 + 180 * scores

    scatter = plt.scatter(
        feature_space[:, 0],
        feature_space[:, 1],
        s=sizes,
        c=scores,
        cmap="viridis",
        alpha=0.8,
        linewidths=0.4,
        edgecolors="none",
        zorder=1,
        label="Model Score",
    )

    positive = dataset_labels[:, dataset_label_index] == 1
    # Use the biggest size of a datapoint as reference for the circle
    max_size = np.max(sizes)

    plt.scatter(
        feature_space[positive, 0],
        feature_space[positive, 1],
        s=max_size,
        #c="red",
        alpha=0.85,
        facecolors='none', edgecolors='r',
        linewidths=2,
        zorder=2,
        label="Ground-truth positive",
    )

    plt.colorbar(scatter, label=f"Model score: {label_name}")
    plt.title(f"{projection_method} Projection Colored by {label_name} Score")
    plt.xlabel(f"{projection_method} 1")
    plt.ylabel(f"{projection_method} 2")
    plt.legend()
    plt.show()

In [38]:
create_viewer(downprojector, visualize_comparison_groundtruth_with_prediction)

### LIME

In [39]:
from lime import lime_image
import torch.nn.functional as F
from skimage.segmentation import mark_boundaries

In [40]:
def batch_predict(images, model = DENSENET121_RES244_CHEX_MODEL):
    model.eval()
   
    if images.ndim == 4 and images.shape[-1] == 3:
        images = images.mean(axis=-1)

    batch = torch.from_numpy(images)
    batch = batch.to(DEVICE)
    score = calculate_model_predictions_score(batch, model)
    new_prection_list = np.zeros((batch.shape[0], 4))
    for selectedDatasetIndex in DATASET_TO_MODEL_INDICES:
        selectedModelIndex = DATASET_TO_MODEL_INDICES[selectedDatasetIndex]
        new_prection_list[:, selectedDatasetIndex] = score[:, selectedModelIndex]
    score = new_prection_list
    return score

In [51]:
def visualize_lime_result(explanation, top_label_index, num_features=10):
    plt.figure()
    temp, mask = explanation.get_image_and_mask(explanation.top_labels[top_label_index], positive_only=False, num_features=num_features, hide_rest=False)
    img_boundry = mark_boundaries(temp/255.0, mask)
    plt.imshow(img_boundry)
    plt.show()

In [58]:
class LimeViewer:

    def __init__(self):
        self.explainer = lime_image.LimeImageExplainer()
        self.explanation = None

    def compute_explanation(self, sample):
        img = sample['img']
        img_rgb = np.stack([img.squeeze()] * 3, axis=-1)

        explanation = self.explainer.explain_instance(img_rgb,
                                                batch_predict,
                                                [i for i in range(len(PATHOLOGIES))],
                                                top_labels=len(PATHOLOGIES), 
                                            
                                                num_samples=1000) 
        return explanation

    def show(self):
        
        plot_lime_output = widgets.Output()

        plot_original_output = widgets.Output()

        lime_status_output = widgets.Output()

        plot_button = widgets.Button(
            description="Explain",
            button_style="primary",
            icon="refresh"
        )

        pathology_dropdown = widgets.Dropdown(
            options=PATHOLOGIES,
            value=PATHOLOGIES[0],
            description="Pathology:",
            layout=widgets.Layout(width="350px")
        )
        dataset_options = [sample['idx'] for sample in NIH_GOOGLE_DATASET]

        sample_dropdown = widgets.Dropdown(
            options=dataset_options,
            value=0,
            description="Sample Index:",
            layout=widgets.Layout(width="350px")
        )

        sample_info = widgets.Label(
            value="Groundtruth PATHOLOGIES: "
        )

        model_score_info = widgets.Label(
            value="Model Score:"
        )

        def update_sample_info(_=None):
            sampleIndex = sample_dropdown.value
            
            sample_label = NIH_GOOGLE_LABELS_LIST[sampleIndex]
            
            sample = NIH_GOOGLE_DATASET[sampleIndex]
            
            sample_info_text = ','.join([DATASET_INDICES_LABEL[l] for l in range(len(sample_label)) if sample_label[l] == 1])
            sample_info.value = "Groundtruth PATHOLOGIES: " + str(sample_info_text)

        def generate_explanation(sample):
            with lime_status_output:
                clear_output(wait=True)

                explanation = self.compute_explanation(sample)
                return explanation

        def update_lime_output(_=None):
            pathology = pathology_dropdown.value  
            explanation = self.explanation
            with plot_lime_output:
                clear_output(wait=True)
            
                if explanation is not None:
                    top_label_index = PATHOLOGIES.index(pathology)
                
                    visualize_lime_result(explanation, top_label_index)

        def create_plot(_=None):

            sampleIndex = sample_dropdown.value

            sample_label = NIH_GOOGLE_LABELS_LIST[sampleIndex]

            sample = NIH_GOOGLE_DATASET[sampleIndex]

            update_sample_info()

            moodel_score_text = ','.join([DATASET_INDICES_LABEL[index] + ": " + str(l) for index, l in enumerate(MODEL_SCORE_PREDICTIONS_CONVERTED[sampleIndex])
                                          if index in DATASET_INDICES_LABEL])
            model_score_info.value = "Model Score:" + str(moodel_score_text)

            self.explanation = generate_explanation(sample)

            update_lime_output()

                
            with plot_original_output:
                clear_output(wait=True)

                plt.figure()
                plt.imshow(sample['img'].squeeze(0), cmap='gray')
                plt.show()
            
        plot_button.on_click(create_plot)
        pathology_dropdown.observe(update_lime_output, names='value')
        sample_dropdown.observe(update_sample_info, names='value')

        display(
        widgets.VBox([
            widgets.HBox([
                pathology_dropdown
            ]),
            widgets.HBox([
                            sample_dropdown,
                            sample_info
                        ]),
            plot_button,
            model_score_info,
            lime_status_output,
             widgets.HBox([
                 plot_original_output,
                 plot_lime_output
             ])
            
        ])
    )

In [ ]:
limeViewer = LimeViewer()
limeViewer.show()